# Sử dụng mô hình LightGBM kết hợp Recursive Forecasting để dự báo doanh thu

### Khai báo các thư viện cần thiết


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

### Lây dữ liệu

In [2]:
path = r'C:\Users\ACER\Documents\datathon-2026-round-1\Datathon_Contest_UITUT\Data' ##<- Thay đổi đường dẫn đến thư mục chứa file CSV của bạn

# 2. Lấy danh sách tất cả các file trong thư mục
files = [f for f in os.listdir(path) if f.endswith('.csv')]

for file in files:
    # Tách tên file và phần mở rộng (vd: 'data.csv' -> 'data')
    file_name = os.path.splitext(file)[0]
    
    # Tạo đường dẫn đầy đủ
    full_path = os.path.join(path, file)
    
    # Đọc file và gán vào biến có tên df_tên_file
    # globals() giúp tạo biến động trong môi trường hiện tại
    globals()[f"df_{file_name}"] = pd.read_csv(full_path)
    
    print(f"Đã load xong: df_{file_name}")

Đã load xong: df_customers
Đã load xong: df_geography
Đã load xong: df_inventory
Đã load xong: df_orders


C:\Users\ACER\AppData\Local\Temp\ipykernel_47308\3693396531.py:15: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[f"df_{file_name}"] = pd.read_csv(full_path)


Đã load xong: df_order_items
Đã load xong: df_payments
Đã load xong: df_products
Đã load xong: df_promotions
Đã load xong: df_returns
Đã load xong: df_reviews
Đã load xong: df_sales
Đã load xong: df_sample_submission
Đã load xong: df_shipments
Đã load xong: df_web_traffic


In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD DỮ LIỆU (Thay đường dẫn của bạn vào đây)
# ============================================================
# Giả sử bạn đã có df_sales và df_web_traffic trong môi trường
# df_sales = pd.read_csv("sales.csv", parse_dates=["Date"])
# df_web_traffic = pd.read_csv("web_traffic.csv", parse_dates=["date"])

def build_expert_data(df_sales, df_web_traffic):
    sales = df_sales.copy()
    sales['Date'] = pd.to_datetime(sales['Date'])
    
    # Gom nhóm Web Traffic theo ngày
    web = df_web_traffic.groupby('date')['sessions'].sum().reset_index().rename(columns={'date':'Date'})
    web['Date'] = pd.to_datetime(web['Date'])
    
    # Tạo khung thời gian đầy đủ từ quá khứ đến hết tháng 6/2024
    all_dates = pd.date_range(start=sales['Date'].min(), end='2024-07-01', freq='D')
    df = pd.DataFrame({'Date': all_dates})
    
    # Merge dữ liệu
    df = pd.merge(df, sales[['Date', 'Revenue', 'COGS']], on='Date', how='left')
    df = pd.merge(df, web, on='Date', how='left')
    
    # Fill dữ liệu trống (Traffic dùng ffill, Revenue/COGS để NaN để xử lý sau)
    df['sessions'] = df['sessions'].ffill().fillna(0)
    
    # Target: Log Transformation để ổn định phương sai
    df['log_rev'] = np.log1p(df['Revenue'].fillna(0))
    df['log_cogs'] = np.log1p(df['COGS'].fillna(0))
    
    # --- Feature Engineering ---
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dow'] = df['Date'].dt.dayofweek
    df['is_weekend'] = (df['dow'] >= 5).astype(int)
    df['is_payday'] = df['day'].isin([15, 30, 31]).astype(int)
    df['is_double_date'] = (df['day'] == df['month']).astype(int)
    
    # Fourier Features (Chu kỳ năm)
    doy = df['Date'].dt.dayofyear
    df['sin_year'] = np.sin(2 * np.pi * doy / 365.25)
    df['cos_year'] = np.cos(2 * np.pi * doy / 365.25)
    
    # Khởi tạo các cột Lag (sẽ được cập nhật trong quá trình đệ quy)
    for lag in [1, 2, 7, 14, 30]:
        df[f'rev_lag_{lag}'] = df['log_rev'].shift(lag).fillna(0)
        df[f'cogs_lag_{lag}'] = df['log_cogs'].shift(lag).fillna(0)
        
    return df

df_final = build_expert_data(df_sales, df_web_traffic)

# ============================================================
# 2. SETUP FEATURES & TRAINING
# ============================================================
FEATURES = [
    'month', 'day', 'dow', 'is_weekend', 'is_payday', 'is_double_date', 
    'sessions', 'sin_year', 'cos_year',
    'rev_lag_1', 'rev_lag_7', 'rev_lag_30',
    'cogs_lag_1', 'cogs_lag_7'
]

# Chia tập dữ liệu
train_mask = df_final['Date'] < '2022-01-01'
val_mask = (df_final['Date'] >= '2022-01-01') & (df_final['Date'] <= '2022-12-31')
test_mask = df_final['Date'] >= '2023-01-01'

train_df = df_final[train_mask]
val_df = df_final[val_mask]

def train_model(target):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.02,
        'num_leaves': 31,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'seed': 42,
        'verbosity': -1
    }
    dtrain = lgb.Dataset(train_df[FEATURES], label=train_df[target])
    dval = lgb.Dataset(val_df[FEATURES], label=val_df[target], reference=dtrain)
    
    model = lgb.train(
        params, dtrain, num_boost_round=2000,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    return model

print("Đang huấn luyện mô hình Revenue...")
model_rev = train_model('log_rev')
print("Đang huấn luyện mô hình COGS...")
model_cogs = train_model('log_cogs')

# ============================================================
# 3. DỰ BÁO ĐỆ QUY (RECURSIVE FORECASTING)
# ============================================================
# Tạo bản sao cho tập test để cập nhật lag
full_data = df_final.copy()
test_dates = full_data[test_mask]['Date'].unique()

print("\nBắt đầu dự báo đệ quy cho giai đoạn 2023 - 2024...")

for current_date in sorted(test_dates):
    # Lấy index của ngày hiện tại
    idx = full_data[full_data['Date'] == current_date].index[0]
    
    # Dự báo log_rev và log_cogs
    current_features = full_data.loc[[idx], FEATURES]
    pred_rev = model_rev.predict(current_features)[0]
    pred_cogs = model_cogs.predict(current_features)[0]
    
    # Ghi đè kết quả vào dataframe
    full_data.loc[idx, 'log_rev'] = pred_rev
    full_data.loc[idx, 'log_cogs'] = pred_cogs
    
    # Cập nhật giá trị Lag cho các ngày tương lai
    for lag in [1, 2, 7, 14, 30]:
        future_date = current_date + pd.Timedelta(days=lag)
        future_idx = full_data[full_data['Date'] == future_date].index
        if not future_idx.empty:
            full_data.loc[future_idx, f'rev_lag_{lag}'] = pred_rev
            full_data.loc[future_idx, f'cogs_lag_{lag}'] = pred_cogs

# ============================================================
# 4. EXPORT SUBMISSION
# ============================================================
# Chuyển ngược từ Log về giá trị gốc
full_data['Revenue'] = np.expm1(full_data['log_rev'])
full_data['COGS'] = np.expm1(full_data['log_cogs'])

# Lọc lấy kết quả cần nộp
submission_data = full_data[test_mask][['Date', 'Revenue', 'COGS']]

# Làm tròn và xử lý ngày tháng
submission_data['Date'] = submission_data['Date'].dt.strftime('%Y-%m-%d')
submission_data['Revenue'] = submission_data['Revenue'].clip(lower=0).round(2)
submission_data['COGS'] = submission_data['COGS'].clip(lower=0).round(2)

# Lưu file
submission_data.to_csv('submission_recursive_final.csv', index=False)

print("\n--- HOÀN THÀNH ---")
print(f"File đã được lưu tại: submission_recursive_final.csv")
print(submission_data.head())

Đang huấn luyện mô hình Revenue...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[775]	training's rmse: 0.122575	valid_1's rmse: 0.250774
Đang huấn luyện mô hình COGS...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[712]	training's rmse: 0.128378	valid_1's rmse: 0.255268

Bắt đầu dự báo đệ quy cho giai đoạn 2023 - 2024...

--- HOÀN THÀNH ---
File đã được lưu tại: submission_recursive_final.csv
            Date     Revenue       COGS
3833  2023-01-01  1198679.70  942041.16
3834  2023-01-02   787770.97  633200.17
3835  2023-01-03   717879.99  589245.46
3836  2023-01-04   797762.21  637700.05
3837  2023-01-05   846993.22  705335.32
